# Shared-fold cross-validation for PCA baselines and QAE

This notebook makes all models use the **exact same outer CV splits**. Preprocessing is fit **inside each fold** to avoid leakage.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "preprocessing_cv_experiments.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# print("Project root:", PROJECT_ROOT)

Project root: /home/senachioiu/AQA-QDL/AQA_QDL-main


In [ ]:
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.svm import OneClassSVM, SVC
from sklearn.linear_model import LogisticRegression
from qiskit.circuit.library import zz_feature_map
from qiskit_machine_learning.kernels import FidelityQuantumKernel


from preprocessing_cv_experiments import (
    load_kdd99,
    load_cic_iot23,
    kdd99_binary_label_map,
    benign_binary_label_map,
    make_fixed_stratified_subset,
    make_shared_kfolds,
    FoldPreprocessorConfig,
    FoldPreprocessor,
    run_linear_pca2_on_shared_folds,
    run_svm_pca4_on_shared_folds,
    summarize_metrics,
    compute_metrics,
    make_inner_validation_split,
)


from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score,
    f1_score, 
    roc_auc_score,
    confusion_matrix, 
    classification_report,
)


from qae import QAEConfig, QuantumAutoencoder

In [ ]:
DATASET = "kdd99"   

KDD_PERCENT10 = True

CIC_PATH = PROJECT_ROOT / "data" / "Merged01.csv"
CIC_LABEL_COL = "Label"
CIC_BENIGN_VALUES = ("benign",)

SUBSET_SIZE = 15000
N_SPLITS = 5
CV_SEED = 42

PREPROC_CONFIG = FoldPreprocessorConfig(
    n_components=4,
    categorical_cols=None,
    drop_cols=None,
    clip_value=1e12,
    balance_train=True,
    random_state=42,
)

QAE_CONFIG = QAEConfig(
    n_qubits=4,
    n_latent=2,
    n_layers=1,
    device_name="default.qubit",
    seed=42,
    use_swap_test_loss=True,
    use_log_cost=True,
)

QAE_TRAIN_STEPS = 40
QAE_LR = 0.03
QAE_BATCH_SIZE = 32
QAE_VAL_FRACTION = 0.2

MAX_QAE_TRAIN_SAMPLES = 6000
MAX_QAE_VAL_SAMPLES = 1000

MAX_QAE_SVM_TRAIN_SAMPLES = None
MAX_QAE_SVM_TEST_SAMPLES = None

rng_global = np.random.default_rng(42)

In [ ]:
if DATASET == "kdd99":
    df = load_kdd99(percent10=KDD_PERCENT10)
    label_col = "labels"
    label_map_fn = kdd99_binary_label_map
elif DATASET == "cic_iot23":
    df = load_cic_iot23(CIC_PATH)
    label_col = CIC_LABEL_COL
    label_map_fn = benign_binary_label_map(CIC_BENIGN_VALUES)
else:
    raise ValueError(f"Unknown DATASET: {DATASET}")

df = df.copy()
df[label_col] = label_map_fn(df[label_col]).astype(int)

X_df_full = df.drop(columns=[label_col])
y_full = df[label_col].to_numpy()

X_df, y = make_fixed_stratified_subset(
    X_df_full,
    y_full,
    total_size=SUBSET_SIZE,
    random_state=CV_SEED,
)

print("Dataset:", DATASET)
print("Raw full shape:", X_df_full.shape)
print("Fixed pooled subset shape:", X_df.shape)
print("Subset class counts:", pd.Series(y).value_counts().to_dict())

Dataset: kdd99
Raw full shape: (494021, 41)
Fixed pooled subset shape: (15000, 41)
Subset class counts: {1: 12046, 0: 2954}


In [ ]:
folds = make_shared_kfolds(
    y=y,
    n_splits=N_SPLITS,
    random_state=CV_SEED,
    shuffle=True,
)

print(f"Total outer folds: {len(folds)}")
print("Example fold sizes:", len(folds[0][1]), len(folds[0][2]))

Total outer folds: 5
Example fold sizes: 12000 3000


## Classical baselines on the exact shared folds

In [6]:

linear_results = run_linear_pca2_on_shared_folds(
    X_df=X_df,
    y=y,
    folds=folds,
    preproc_config=PREPROC_CONFIG,
)

svm_results = run_svm_pca4_on_shared_folds(
    X_df=X_df,
    y=y,
    folds=folds,
    preproc_config=PREPROC_CONFIG,
)

classical_results = pd.concat([linear_results, svm_results], ignore_index=True)
classical_results.head()


,fold,model,accuracy,precision,recall,f1,roc_auc
0,1,PCA-2 + LogisticRegression,0.970667,0.982143,0.981328,0.981735,0.993327
1,2,PCA-2 + LogisticRegression,0.976000,0.989527,0.980490,0.984987,0.993786
2,3,PCA-2 + LogisticRegression,0.975667,0.989933,0.979660,0.984769,0.993773
3,4,PCA-2 + LogisticRegression,0.977000,0.989540,0.981735,0.985622,0.993668
4,5,PCA-2 + LogisticRegression,0.978333,0.992851,0.980075,0.986422,0.995934


In [7]:

classical_summary = (
    classical_results
    .groupby("model", as_index=False)
    .apply(lambda g: summarize_metrics(g))
    .reset_index(level=0)
    .rename(columns={"level_0": "group_idx"})
)

for model_name in classical_results["model"].unique():
    print("\n", model_name)
    display(summarize_metrics(classical_results[classical_results["model"] == model_name])[["metric", "mean ± std"]])



 PCA-2 + LogisticRegression


,metric,mean ± std
0,accuracy,0.9755 ± 0.0029
1,precision,0.9888 ± 0.0040
2,recall,0.9807 ± 0.0009
3,f1,0.9847 ± 0.0018
4,roc_auc,0.9941 ± 0.0010



 PCA-4 + SVM


,metric,mean ± std
0,accuracy,0.9857 ± 0.0017
1,precision,1.0000 ± 0.0000
2,recall,0.9822 ± 0.0021
3,f1,0.9910 ± 0.0011
4,roc_auc,0.9942 ± 0.0015


## QAE + SVM on the exact same outer folds

In [ ]:
def maybe_subsample(X, y, max_n, rng):
    if max_n is None or len(X) <= max_n:
        return X, y
    idx = rng.choice(len(X), size=max_n, replace=False)
    return X[idx], y[idx]


def run_qae_svm_on_shared_folds(
    X_df,
    y,
    folds,
    preproc_config,
    qae_config,
    train_steps=40,
    lr=0.03,
    batch_size=32,
    val_fraction=0.2,
    max_qae_train_samples=6000,
    max_qae_val_samples=1000,
    max_svm_train_samples=None,
    max_svm_test_samples=None,
):
    rows = []

    for fold_id, train_idx, test_idx in folds:
        print(f"\n=== QAE fold {fold_id + 1}/{len(folds)} ===")

        fold_seed = 1000 + fold_id + 42
        rng = np.random.default_rng(fold_seed)

        fp = FoldPreprocessor(
            FoldPreprocessorConfig(
                n_components=preproc_config.n_components,
                categorical_cols=preproc_config.categorical_cols,
                drop_cols=preproc_config.drop_cols,
                clip_value=preproc_config.clip_value,
                balance_train=preproc_config.balance_train,
                random_state=fold_seed,
            )
        )
        train_pack = fp.fit_transform(X_df.iloc[train_idx], y[train_idx])
        test_pack = fp.transform(X_df.iloc[test_idx])

        X_train_full = train_pack["X_train_pca"]
        y_train_full = train_pack["y_train"]
        X_test_full = test_pack["X_test_pca"]
        y_test_full = y[test_idx]

        inner_train_idx, val_idx = make_inner_validation_split(
            y_train_full,
            test_size=val_fraction,
            random_state=fold_seed,
        )

        X_train_qae = X_train_full[inner_train_idx]
        y_train_qae = y_train_full[inner_train_idx]
        X_val_qae = X_train_full[val_idx]
        y_val_qae = y_train_full[val_idx]

        X_train_qae, y_train_qae = maybe_subsample(X_train_qae, y_train_qae, max_qae_train_samples, rng)
        X_val_qae, y_val_qae = maybe_subsample(X_val_qae, y_val_qae, max_qae_val_samples, rng)

        qae_cfg_fold = QAEConfig(
            n_qubits=qae_config.n_qubits,
            n_latent=qae_config.n_latent,
            n_layers=qae_config.n_layers,
            device_name=qae_config.device_name,
            seed=fold_seed,
            use_swap_test_loss=qae_config.use_swap_test_loss,
            use_log_cost=qae_config.use_log_cost,
            eps=qae_config.eps,
        )
        qae = QuantumAutoencoder(qae_cfg_fold)

        best_params, history, scaler = qae.train(
            X_train_raw=X_train_qae,
            X_val_raw=X_val_qae,
            steps=train_steps,
            lr=lr,
            batch_size=batch_size,
            val_every=max(5, train_steps // 8),
            verbose=False,
        )

        X_svm_train = X_train_full
        y_svm_train = y_train_full
        X_svm_test = X_test_full
        y_svm_test = y_test_full

        X_svm_train, y_svm_train = maybe_subsample(X_svm_train, y_svm_train, max_svm_train_samples, rng)
        X_svm_test, y_svm_test = maybe_subsample(X_svm_test, y_svm_test, max_svm_test_samples, rng)

        Z_train = qae.batch_transform_latent_features_from_raw(
            X_svm_train, best_params, scaler, batch_size=1000, verbose=False
        )
        Z_test = qae.batch_transform_latent_features_from_raw(
            X_svm_test, best_params, scaler, batch_size=1000, verbose=False
        )

        svm_clf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=fold_seed)
        svm_clf.fit(Z_train, y_svm_train)
        y_pred = svm_clf.predict(Z_test)
        y_prob = svm_clf.predict_proba(Z_test)[:, 1]

        row = {"fold": fold_id + 1, "model": "QAE(latent=2) + SVM"}
        row.update(compute_metrics(y_svm_test, y_pred, y_prob))
        rows.append(row)

    return pd.DataFrame(rows)

In [9]:

qae_results = run_qae_svm_on_shared_folds(
    X_df=X_df,
    y=y,
    folds=folds,
    preproc_config=PREPROC_CONFIG,
    qae_config=QAE_CONFIG,
    train_steps=QAE_TRAIN_STEPS,
    lr=QAE_LR,
    batch_size=QAE_BATCH_SIZE,
    val_fraction=QAE_VAL_FRACTION,
    max_qae_train_samples=MAX_QAE_TRAIN_SAMPLES,
    max_qae_val_samples=MAX_QAE_VAL_SAMPLES,
    max_svm_train_samples=MAX_QAE_SVM_TRAIN_SAMPLES,
    max_svm_test_samples=MAX_QAE_SVM_TEST_SAMPLES,
)

qae_results.head()



=== QAE fold 1/5 ===

=== QAE fold 2/5 ===

=== QAE fold 3/5 ===

=== QAE fold 4/5 ===

=== QAE fold 5/5 ===


,fold,model,accuracy,precision,recall,f1,roc_auc
0,1,QAE(latent=2) + SVM,0.940333,0.999552,0.926141,0.961447,0.991332
1,2,QAE(latent=2) + SVM,0.944333,0.999554,0.931092,0.964109,0.985922
2,3,QAE(latent=2) + SVM,0.936333,1.000000,0.920714,0.958721,0.991089
3,4,QAE(latent=2) + SVM,0.947333,1.000000,0.934413,0.966094,0.986032
4,5,QAE(latent=2) + SVM,0.966667,0.998704,0.959734,0.978831,0.986517


In [10]:

all_results = pd.concat([classical_results, qae_results], ignore_index=True)

for model_name in all_results["model"].unique():
    print("\n", model_name)
    display(summarize_metrics(all_results[all_results["model"] == model_name])[["metric", "mean ± std"]])



 PCA-2 + LogisticRegression


,metric,mean ± std
0,accuracy,0.9755 ± 0.0029
1,precision,0.9888 ± 0.0040
2,recall,0.9807 ± 0.0009
3,f1,0.9847 ± 0.0018
4,roc_auc,0.9941 ± 0.0010



 PCA-4 + SVM


,metric,mean ± std
0,accuracy,0.9857 ± 0.0017
1,precision,1.0000 ± 0.0000
2,recall,0.9822 ± 0.0021
3,f1,0.9910 ± 0.0011
4,roc_auc,0.9942 ± 0.0015



 QAE(latent=2) + SVM


,metric,mean ± std
0,accuracy,0.9470 ± 0.0117
1,precision,0.9996 ± 0.0005
2,recall,0.9344 ± 0.0151
3,f1,0.9658 ± 0.0078
4,roc_auc,0.9882 ± 0.0028


In [11]:

summary_table = []
for model_name in all_results["model"].unique():
    sm = summarize_metrics(all_results[all_results["model"] == model_name])
    row = {"model": model_name}
    for _, r in sm.iterrows():
        row[r["metric"]] = r["mean ± std"]
    summary_table.append(row)

summary_table = pd.DataFrame(summary_table)
summary_table


,model,accuracy,precision,recall,f1,roc_auc
0,PCA-2 + LogisticRegression,0.9755 ± 0.0029,0.9888 ± 0.0040,0.9807 ± 0.0009,0.9847 ± 0.0018,0.9941 ± 0.0010
1,PCA-4 + SVM,0.9857 ± 0.0017,1.0000 ± 0.0000,0.9822 ± 0.0021,0.9910 ± 0.0011,0.9942 ± 0.0015
2,QAE(latent=2) + SVM,0.9470 ± 0.0117,0.9996 ± 0.0005,0.9344 ± 0.0151,0.9658 ± 0.0078,0.9882 ± 0.0028


## Model comparison

In [ ]:
def evaluate_kernel_batched(qkernel, X, Y=None, batch_size=25, desc="Kernel"):
    if Y is None:
        Y = X
    blocks = []
    for start in tqdm(range(0, len(X), batch_size), desc=desc):
        end = min(start + batch_size, len(X))
        K_chunk = qkernel.evaluate(x_vec=X[start:end], y_vec=Y)
        blocks.append(K_chunk)
    return np.vstack(blocks)


def subsample_balanced_by_index(y, n_samples, rng):
    y = np.asarray(y)
    classes = np.unique(y)
    n_per_class = n_samples // len(classes)
    indices = []
    for c in classes:
        c_idx = np.where(y == c)[0]
        chosen = rng.choice(c_idx, size=min(n_per_class, len(c_idx)), replace=False)
        indices.append(chosen)
    indices = np.concatenate(indices)
    rng.shuffle(indices)
    return indices


def _run_one_fold(
    X_df, y, fold, preproc_config, qae_config,
    n_train, n_test,
    qae_steps, qae_lr, qae_batch_size, qae_val_fraction,
    max_qae_train, max_qae_val,
    qsvm_reps, qsvm_nu, qsvm_kernel_batch,
    random_state,
):
    fold_id, train_idx, test_idx = fold
    fold_seed = 1000 + fold_id + random_state
    rng = np.random.default_rng(fold_seed)

    fp = FoldPreprocessor(
        FoldPreprocessorConfig(
            n_components=preproc_config.n_components,
            categorical_cols=preproc_config.categorical_cols,
            drop_cols=preproc_config.drop_cols,
            clip_value=preproc_config.clip_value,
            balance_train=preproc_config.balance_train,
            random_state=fold_seed,
        )
    )
    train_pack = fp.fit_transform(X_df.iloc[train_idx], y[train_idx])
    test_pack = fp.transform(X_df.iloc[test_idx])

    X_train_pca = train_pack["X_train_pca"]
    y_train_full = train_pack["y_train"]
    X_test_pca = test_pack["X_test_pca"]
    y_test_full = y[test_idx]

    #  QAE train
    inner_tr, inner_val = make_inner_validation_split(
        y_train_full, test_size=qae_val_fraction, random_state=fold_seed,
    )
    X_qae_tr = X_train_pca[inner_tr]
    X_qae_val = X_train_pca[inner_val]

    if max_qae_train and len(X_qae_tr) > max_qae_train:
        idx = rng.choice(len(X_qae_tr), size=max_qae_train, replace=False)
        X_qae_tr = X_qae_tr[idx]
    if max_qae_val and len(X_qae_val) > max_qae_val:
        idx = rng.choice(len(X_qae_val), size=max_qae_val, replace=False)
        X_qae_val = X_qae_val[idx]

    qae_cfg = QAEConfig(
        n_qubits=qae_config.n_qubits,
        n_latent=qae_config.n_latent,
        n_layers=qae_config.n_layers,
        device_name=qae_config.device_name,
        seed=fold_seed,
        use_swap_test_loss=qae_config.use_swap_test_loss,
        use_log_cost=qae_config.use_log_cost,
        eps=qae_config.eps,
    )
    qae = QuantumAutoencoder(qae_cfg)
    best_params, history, scaler = qae.train(
        X_train_raw=X_qae_tr, X_val_raw=X_qae_val,
        steps=qae_steps, lr=qae_lr, batch_size=qae_batch_size,
        val_every=max(5, qae_steps // 8), verbose=False,
    )

    # project to latent space using QAE
    Z_train_full = qae.batch_transform_latent_features_from_raw(
        X_train_pca, best_params, scaler, batch_size=1000, verbose=False,
    )
    Z_test_full = qae.batch_transform_latent_features_from_raw(
        X_test_pca, best_params, scaler, batch_size=1000, verbose=False,
    )

    train_sel = subsample_balanced_by_index(y_train_full, n_train, rng)
    test_sel  = subsample_balanced_by_index(y_test_full,  n_test,  rng)

    X_tr = X_train_pca[train_sel]
    X_te = X_test_pca[test_sel]
    Z_tr = Z_train_full[train_sel]
    Z_te = Z_test_full[test_sel]
    y_tr = y_train_full[train_sel]
    y_te = y_test_full[test_sel]

    # run each model on same data
    rows = []

    def _metrics(name, yt, yp, yprob, elapsed):
        return dict(
            fold=fold_id + 1,
            model=name,
            accuracy=accuracy_score(yt, yp),
            precision=precision_score(yt, yp, zero_division=0),
            recall=recall_score(yt, yp, zero_division=0),
            f1=f1_score(yt, yp, zero_division=0),
            roc_auc=(
                roc_auc_score(yt, yprob)
                if yprob is not None and len(np.unique(yt)) > 1
                else np.nan
            ),
            time_sec=elapsed,
        )

    # Logistic
    t0 = time.time()
    lr_clf = LogisticRegression(max_iter=1000, random_state=fold_seed)
    lr_clf.fit(X_tr, y_tr)
    rows.append(_metrics(
        "Logistic Regression (PCA)", y_te,
        lr_clf.predict(X_te), lr_clf.predict_proba(X_te)[:, 1], time.time() - t0,
    ))

    # SVM
    t0 = time.time()
    svm_clf = SVC(kernel="rbf", C=1.0, gamma="scale",
                  probability=True, random_state=fold_seed)
    svm_clf.fit(X_tr, y_tr)
    rows.append(_metrics(
        "Classical SVM (PCA)", y_te,
        svm_clf.predict(X_te), svm_clf.predict_proba(X_te)[:, 1], time.time() - t0,
    ))

    # QAE + Classical SVM
    t0 = time.time()
    qsvm_clf = SVC(kernel="rbf", C=1.0, gamma="scale",
                   probability=True, random_state=fold_seed)
    qsvm_clf.fit(Z_tr, y_tr)
    rows.append(_metrics(
        "QAE + Classical SVM", y_te,
        qsvm_clf.predict(Z_te), qsvm_clf.predict_proba(Z_te)[:, 1], time.time() - t0,
    ))

    # QAE + Quantum OC-SVM
    t0 = time.time()
    normal_mask = (y_tr == 0)
    Z_train_normal = Z_tr[normal_mask]

    feature_dim = Z_tr.shape[1]
    fm = zz_feature_map(feature_dimension=feature_dim, reps=qsvm_reps)
    qkernel = FidelityQuantumKernel(feature_map=fm)

    K_train = evaluate_kernel_batched(
        qkernel, Z_train_normal, Y=Z_train_normal,
        batch_size=qsvm_kernel_batch, desc=f"QK train f{fold_id+1}",
    )
    K_test = evaluate_kernel_batched(
        qkernel, Z_te, Y=Z_train_normal,
        batch_size=qsvm_kernel_batch, desc=f"QK test  f{fold_id+1}",
    )

    ocsvm = OneClassSVM(kernel="precomputed", nu=qsvm_nu)
    ocsvm.fit(K_train)
    ocsvm_raw = ocsvm.predict(K_test)
    ocsvm_pred = np.where(ocsvm_raw == 1, 0, 1)

    rows.append(_metrics(
        "QAE + Quantum OC-SVM", y_te,
        ocsvm_pred, None, time.time() - t0,
    ))

    return rows, dict(qae=qae, best_params=best_params, scaler=scaler,
                      X_tr_pca=X_tr, X_te_pca=X_te, y_tr=y_tr, y_te=y_te)



def run_4model_cv(
    X_df, y, folds, preproc_config, qae_config,
    n_train=100, n_test=100,
    qae_steps=40, qae_lr=0.03, qae_batch_size=32,
    qae_val_fraction=0.2,
    max_qae_train=6000, max_qae_val=1000,
    qsvm_reps=1, qsvm_nu=0.1, qsvm_kernel_batch=20,
    random_state=42,
):
    all_rows = []
    last_artifacts = None

    for i, fold in enumerate(folds):
        print(f"\n{'=' * 60}")
        print(f"  FOLD {i + 1} / {len(folds)}")
        print(f"{'=' * 60}")
        rows, artifacts = _run_one_fold(
            X_df, y, fold, preproc_config, qae_config,
            n_train, n_test,
            qae_steps, qae_lr, qae_batch_size, qae_val_fraction,
            max_qae_train, max_qae_val,
            qsvm_reps, qsvm_nu, qsvm_kernel_batch,
            random_state,
        )
        all_rows.extend(rows)
        last_artifacts = artifacts
        print(f"  Fold {i + 1} done.")

    per_fold_df = pd.DataFrame(all_rows)

    # get mean +- std
    metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "time_sec"]
    summary_rows = []
    for model_name in per_fold_df["model"].unique():
        mdf = per_fold_df[per_fold_df["model"] == model_name]
        row = {"model": model_name}
        for m in metric_cols:
            mean = mdf[m].mean()
            std  = mdf[m].std(ddof=1)
            row[f"{m}_mean"] = mean
            row[f"{m}_std"]  = std
            row[m] = f"{mean:.4f} ± {std:.4f}"
        summary_rows.append(row)
    summary_df = pd.DataFrame(summary_rows)

    print(f"\n{'=' * 60}")
    print(f"  ALL {len(folds)} FOLDS COMPLETE")
    print(f"{'=' * 60}")

    return per_fold_df, summary_df, last_artifacts

In [ ]:
per_fold_df, summary_df, last_artifacts = run_4model_cv(
    X_df=X_df,
    y=y,
    folds=folds,                           
    preproc_config=PREPROC_CONFIG,
    qae_config=QAE_CONFIG,
    n_train=100,                          
    n_test=100,
    qae_steps=QAE_TRAIN_STEPS,
    qae_lr=QAE_LR,
    qae_batch_size=QAE_BATCH_SIZE,
    qae_val_fraction=QAE_VAL_FRACTION,
    max_qae_train=MAX_QAE_TRAIN_SAMPLES,
    max_qae_val=MAX_QAE_VAL_SAMPLES,
    qsvm_reps=1,
    qsvm_nu=0.1,
    qsvm_kernel_batch=20,
    random_state=42,
)


  FOLD 1 / 5


QK train f1:   0%|          | 0/3 [00:00<?, ?it/s]

QK test  f1:   0%|          | 0/5 [00:00<?, ?it/s]

  ✓ Fold 1 done.

  FOLD 2 / 5


QK train f2:   0%|          | 0/3 [00:00<?, ?it/s]

...

QK test  f4:   0%|          | 0/5 [00:00<?, ?it/s]

  ✓ Fold 4 done.

  FOLD 5 / 5


QK train f5:   0%|          | 0/3 [00:00<?, ?it/s]

QK test  f5:   0%|          | 0/5 [00:00<?, ?it/s]

  ✓ Fold 5 done.

  ALL 5 FOLDS COMPLETE


In [ ]:
display(
    summary_df[["model", "accuracy", "precision", "recall", "f1"]]
    .style.hide(axis="index")
)

print("\nPer-fold metrics:")
display(
    per_fold_df
    .style.format({
        "accuracy": "{:.4f}", "precision": "{:.4f}",
        "recall": "{:.4f}", "f1": "{:.4f}",
        "roc_auc": "{:.4f}", "time_sec": "{:.2f}",
    })
    .hide(axis="index")
)

model,accuracy,precision,recall,f1
Logistic Regression (PCA),0.9580 ± 0.0268,0.9881 ± 0.0176,0.9280 ± 0.0626,0.9559 ± 0.0296
Classical SVM (PCA),0.9600 ± 0.0324,0.9960 ± 0.0089,0.9240 ± 0.0684,0.9575 ± 0.0357
QAE + Classical SVM,0.9480 ± 0.0228,1.0000 ± 0.0000,0.8960 ± 0.0456,0.9447 ± 0.0255
QAE + Quantum OC-SVM,0.9040 ± 0.0241,0.8771 ± 0.0184,0.9400 ± 0.0469,0.9070 ± 0.0255



Per-fold metrics:


fold,model,accuracy,precision,recall,f1,roc_auc,time_sec
1,Logistic Regression (PCA),0.9900,1.0000,0.9800,0.9899,0.9996,0.01
1,Classical SVM (PCA),0.9900,1.0000,0.9800,0.9899,1.0000,0.00
1,QAE + Classical SVM,0.9700,1.0000,0.9400,0.9691,0.9768,0.00
1,QAE + Quantum OC-SVM,0.9100,0.8727,0.9600,0.9143,nan,12.67
2,Logistic Regression (PCA),0.9700,0.9608,0.9800,0.9703,0.9944,0.01
2,Classical SVM (PCA),0.9800,0.9800,0.9800,0.9800,0.9820,0.00
2,QAE + Classical SVM,0.9500,1.0000,0.9000,0.9474,0.9884,0.00
2,QAE + Quantum OC-SVM,0.9200,0.8750,0.9800,0.9245,nan,12.71
3,Logistic Regression (PCA),0.9700,0.9796,0.9600,0.9697,0.9888,0.01
3,Classical SVM (PCA),0.9800,1.0000,0.9600,0.9796,0.9792,0.00
